# SAAM Project — Part I, Section 2.3: Value-Weighted Portfolio & Comparison
**Group AT** — North America (AMER) / Scope 1+2

This notebook:
1. Constructs the value-weighted (market-cap weighted) benchmark portfolio
2. Computes its ex-post performance statistics
3. Compares P(vw) with P(mv)_oos: cumulative returns, summary statistics

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import os
CHARTS_DIR = './Charts/'
os.makedirs(CHARTS_DIR, exist_ok=True)

Y0       = 2013
Y_END    = 2024
DATA_DIR = './cleaned_data/'

print('Setup complete.')

## 1. Load Data

In [ ]:
required_files = [
    'returns_monthly.csv', 'mv_monthly.csv', 'investment_sets.csv',
    'risk_free_rate.csv', 'firm_names.csv', 'mv_portfolio_returns.csv'
]
missing = [f for f in required_files
           if not os.path.exists(f'{DATA_DIR}{f}')]
if missing:
    raise FileNotFoundError(
        f"Missing files: {missing}. "
        f"Run data_cleaning.ipynb and min_variance_portfolio.ipynb first."
    )

# Monthly returns
returns_m = pd.read_csv(f'{DATA_DIR}returns_monthly.csv', index_col=0, parse_dates=False)
returns_m.columns = pd.to_datetime(returns_m.columns)

# Monthly market values (in million USD)
mv_m = pd.read_csv(f'{DATA_DIR}mv_monthly.csv', index_col=0, parse_dates=False)
mv_m.columns = pd.to_datetime(mv_m.columns)

# Investment sets
inv_sets_df = pd.read_csv(f'{DATA_DIR}investment_sets.csv')
investment_sets = {int(Y): grp['ISIN'].tolist() for Y, grp in inv_sets_df.groupby('Year')}

# Risk-free rate
rf_df = pd.read_csv(f'{DATA_DIR}risk_free_rate.csv')
rf_dict = {(int(r['year']), int(r['month'])): r['RF_monthly']
           for _, r in rf_df.iterrows()}

# Firm names (as dict for safe .get() calls)
firm_names = pd.read_csv(f'{DATA_DIR}firm_names.csv', index_col=0).squeeze().to_dict()

# Min-variance portfolio returns (from Section 2.2)
df_mv_ret = pd.read_csv(f'{DATA_DIR}mv_portfolio_returns.csv', index_col=0, parse_dates=True)

# Guard: ensure MV returns were generated with the Jan 2026 fix applied.
# If this assert fails, re-run min_variance_portfolio.ipynb first.
assert len(df_mv_ret) == 144, (
    f"MV portfolio returns has {len(df_mv_ret)} months (expected 144). "
    f"Re-run min_variance_portfolio.ipynb after applying the Jan 2026 fix."
)

# Return dates as list
dates_ret = returns_m.columns.tolist()

print(f'Returns: {returns_m.shape[0]} firms × {returns_m.shape[1]} months')
print(f'Market values: {mv_m.shape[0]} firms × {mv_m.shape[1]} months')
print(f'MV portfolio returns loaded: {len(df_mv_ret)} months')

## 2. Align Return and Market Value Dates

The return dates and MV dates may not match exactly (e.g., month-end dates can differ by a day).
We build a mapping from each return date to the corresponding MV date by matching year-month.

In [ ]:
# Build (year, month) → mv column mapping (once, upfront)
mv_date_map = {}
for c in mv_m.columns:
    mv_date_map[(c.year, c.month)] = c

# Build (year, month) → return column mapping
ret_date_map = {}
for c in returns_m.columns:
    ret_date_map[(c.year, c.month)] = c

# Verify alignment
mv_ym = set(mv_date_map.keys())
ret_ym = set(ret_date_map.keys())
common_ym = mv_ym & ret_ym
print(f'MV dates: {len(mv_ym)} months')
print(f'Return dates: {len(ret_ym)} months')
print(f'Common (year,month): {len(common_ym)} months')

missing_in_mv = ret_ym - mv_ym
if missing_in_mv:
    print(f'  ⚠ {len(missing_in_mv)} return months have no matching MV date')

# Verify that Dec 2013 and Dec 2024 market caps are available.
# Dec 2013 is needed to compute Jan 2014 VW weights (first performance month).
# Dec 2024 is needed to compute Jan 2025 VW weights (last performance month).
assert (2013, 12) in mv_date_map, (
    "Dec 2013 market caps missing — cannot compute Jan 2014 VW weights."
)
assert (2024, 12) in mv_date_map, (
    "Dec 2024 market caps missing — cannot compute Jan 2025 VW weights."
)
print("✓ Dec 2013 and Dec 2024 market caps available for VW weight initialization.")

## 3. Value-Weighted Portfolio Construction

$$R^{(vw)}_{t+1} = \sum_{i=1}^{N} w_{i,t} \, R_{i,t+1}$$

where $w_{i,t} = \text{Cap}_{i,t} / \sum_{j=1}^{N} \text{Cap}_{j,t}$.

**Key decisions:**
- VW weights rebalance monthly (using market caps at end of previous month)
- Same investment set as MV portfolio (determined annually at year-end)
- No forward-fill of market caps: if a firm's cap becomes NaN or 0 (default/delist),
  its weight drops to 0 that month. This avoids 'zombie stocks' that would
  artificially suppress portfolio volatility.

In [ ]:
vw_returns = []
vw_weights_yearend = {}  # weights at end of Dec Y, keyed by allocation year Y

for Y in range(Y0, Y_END + 1):
    isins_Y = investment_sets[Y]
    year_next = Y + 1
    
    # Monthly dates in year Y+1 (performance year)
    months_next = [(d.year, d.month) for d in dates_ret if d.year == year_next]
    
    if not months_next:
        continue
    
    for ym in months_next:
        # Previous month's (year, month) for market cap weights (time t)
        prev_month = ym[1] - 1
        prev_year = ym[0]
        if prev_month == 0:
            prev_month = 12
            prev_year -= 1
        ym_prev = (prev_year, prev_month)
        
        # 1. Get EXACT market caps at time t — no forward-fill
        #    If a firm's cap is NaN or 0, it has exited (default/delist)
        #    and its weight must be 0 (no zombie stocks)
        if ym_prev not in mv_date_map:
            print(f'  ⚠ MV data missing for {ym_prev} — month {ym} skipped')
            continue
        
        mv_col = mv_date_map[ym_prev]
        caps = mv_m.loc[isins_Y, mv_col].copy()
        
        # 2. Keep only firms that actually exist with positive market cap
        # Firms that delist mid-year are automatically excluded when their market cap
        # becomes NaN or zero. Their last return was already set to -100% in
        # data_cleaning.ipynb; in subsequent months they simply exit the portfolio.
        caps_clean = caps[caps.notna() & (caps > 0)]
        
        # caps_clean contains only positive values by construction, so sum == 0
        # is impossible if len > 0. The second condition is kept for safety only.
        if len(caps_clean) == 0:
            print(f'  ⚠ No valid market caps for {ym} — month skipped')
            continue
        
        # 3. w_{i,t} = Cap_{i,t} / Sum(Cap_{j,t})
        w_vw = caps_clean / caps_clean.sum()
        
        # 4. Get returns for month t+1
        ret_col = ret_date_map.get(ym)
        if ret_col is None:
            print(f'  ⚠ No return data for {ym} — month skipped')
            continue
        r_stocks = returns_m.loc[w_vw.index, ret_col].fillna(0.0)
        
        # 5. R^{(vw)}_{t+1} = Sum(w_{i,t} * R_{i,t+1})
        r_vw = (w_vw * r_stocks).sum()
        vw_returns.append({'Date': ret_col, 'Return': r_vw})
    
    # Store end-of-year-Y weights (Dec Y caps) for Part II
    # These are the weights at end of allocation year Y
    ym_dec = (Y, 12)
    if ym_dec in mv_date_map:
        mv_dec = mv_date_map[ym_dec]
        caps_dec = mv_m.loc[isins_Y, mv_dec].copy()
        caps_dec = caps_dec[caps_dec.notna() & (caps_dec > 0)]
        if len(caps_dec) > 0:
            # NOTE for Part II: vw_weights_yearend[Y] includes only firms with positive
            # market cap at Dec Y. Firms in investment_sets[Y] with NaN cap are excluded
            # from both the numerator and denominator of CF(vw), which may slightly
            # underestimate the true carbon footprint of the benchmark.
            vw_weights_yearend[Y] = caps_dec / caps_dec.sum()

df_vw_ret = pd.DataFrame(vw_returns).set_index('Date').sort_index()

# Verify sample length matches PDF specification (T=144, Jan 2014–Dec 2025)
assert len(df_vw_ret) == 144, (
    f"Expected 144 months (Jan 2014–Dec 2025), got {len(df_vw_ret)}. "
    f"Last date: {df_vw_ret.index[-1].strftime('%Y-%m')}"
)

n_vw = len(df_vw_ret)
n_mv = len(df_mv_ret)
print(f'VW portfolio: {n_vw} months  |  MV portfolio: {n_mv} months')
if n_vw != n_mv:
    print(f'  ⚠ Month count mismatch between VW and MV portfolios.')
    vw_dates = set(df_vw_ret.index)
    mv_dates = set(df_mv_ret.index)
    only_vw = vw_dates - mv_dates
    only_mv = mv_dates - vw_dates
    if only_vw: print(f'  Only in VW: {sorted(only_vw)}')
    if only_mv: print(f'  Only in MV: {sorted(only_mv)}')
else:
    print('  ✓ Same number of months in both portfolios.')

print(f'Period: {df_vw_ret.index[0].strftime("%Y-%m")} to '
      f'{df_vw_ret.index[-1].strftime("%Y-%m")}')
df_vw_ret.head()

## 4. VW Portfolio Performance Statistics

In [ ]:
r_vw = df_vw_ret['Return'].values

# Excess returns
rf_series_vw = np.array([rf_dict.get((dt.year, dt.month), 0.0) 
                          for dt in df_vw_ret.index])
excess_vw = r_vw - rf_series_vw

# Statistics
mu_ann_vw  = np.mean(r_vw) * 12
vol_ann_vw = np.std(r_vw, ddof=0) * np.sqrt(12)
er_mean_ann_vw = np.mean(excess_vw) * 12
sharpe_vw = er_mean_ann_vw / vol_ann_vw
rf_ann_vw = np.mean(rf_series_vw) * 12
# rf_ann is identical for both portfolios since they cover the same sample period.
# The assert below verifies this explicitly in case of any future sample mismatch.
rf_ann = rf_ann_vw

print('═══════════════════════════════════════════════════════')
print('  VALUE-WEIGHTED PORTFOLIO — P(vw)')
print('═══════════════════════════════════════════════════════')
print(f'  Annualized return (μ̄_p):     {mu_ann_vw:>8.4f}  ({mu_ann_vw*100:.2f}%)')
print(f'  Annualized volatility (σ_p):  {vol_ann_vw:>8.4f}  ({vol_ann_vw*100:.2f}%)')
print(f'  Risk-free rate (avg ann.):     {rf_ann:>8.4f}  ({rf_ann*100:.2f}%)')
print(f'  Sharpe ratio:                  {sharpe_vw:>8.4f}')
print(f'  Min monthly return:            {np.min(r_vw):>8.4f}  ({np.min(r_vw)*100:.2f}%)')
print(f'  Max monthly return:            {np.max(r_vw):>8.4f}  ({np.max(r_vw)*100:.2f}%)')
print('═══════════════════════════════════════════════════════')

## 5. Comparison: Summary Statistics Table

In [ ]:
# MV statistics (recompute from loaded data for consistency)
r_mv = df_mv_ret['Return'].values
rf_series_mv = np.array([rf_dict.get((dt.year, dt.month), 0.0) 
                          for dt in df_mv_ret.index])
excess_mv = r_mv - rf_series_mv

mu_ann_mv  = np.mean(r_mv) * 12
vol_ann_mv = np.std(r_mv, ddof=0) * np.sqrt(12)
er_mean_ann_mv = np.mean(excess_mv) * 12
sharpe_mv = er_mean_ann_mv / vol_ann_mv
rf_ann_mv = np.mean(rf_series_mv) * 12

assert abs(rf_ann_vw - rf_ann_mv) < 1e-8, (
    "Risk-free rate mismatch between VW and MV samples — "
    "check that both portfolios cover exactly Jan 2014–Dec 2025."
)

# Numerical DataFrame for saving to CSV and downstream use in Part II
comparison_num = pd.DataFrame({
    'P(mv)_oos': [mu_ann_mv, vol_ann_mv, sharpe_mv, np.min(r_mv), np.max(r_mv)],
    'P(vw)':     [mu_ann_vw, vol_ann_vw, sharpe_vw, np.min(r_vw), np.max(r_vw)],
}, index=['Ann_Return', 'Ann_Volatility', 'Sharpe_Ratio',
          'Min_Monthly_Return', 'Max_Monthly_Return'])

# Formatted version for display only — not saved
print('\nComparison of Portfolio Strategies (2014–2025)\n')
print(comparison_num.applymap(lambda x: f'{x:.4f}').to_string())

## 6. Cumulative Return Comparison

In [ ]:
cum_mv = (1 + df_mv_ret['Return']).cumprod()
cum_vw = (1 + df_vw_ret['Return']).cumprod()

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(cum_mv.index, cum_mv.values, color='#1f4e79', linewidth=1.5, label='P(mv)_oos')
ax.plot(cum_vw.index, cum_vw.values, color='#c0392b', linewidth=1.5, label='P(vw)')
ax.set_title('Cumulative Returns — Min-Variance vs Value-Weighted (2014–2025)', fontsize=13)
ax.set_xlabel('Date')
ax.set_ylabel('Wealth Index (starting at 1)')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{CHARTS_DIR}mv_vs_vw_cumulative.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Final wealth — P(mv)_oos: {cum_mv.iloc[-1]:.4f}  |  P(vw): {cum_vw.iloc[-1]:.4f}')

## 7. Rolling Volatility Comparison

In [ ]:
# 12-month rolling annualized volatility
# Use ddof=0 for consistency with summary statistics computed via np.std(..., ddof=0)
roll_vol_mv = df_mv_ret['Return'].rolling(12).std(ddof=0) * np.sqrt(12)
roll_vol_vw = df_vw_ret['Return'].rolling(12).std(ddof=0) * np.sqrt(12)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(roll_vol_mv.index, roll_vol_mv.values, color='#1f4e79', linewidth=1.2, label='P(mv)_oos')
ax.plot(roll_vol_vw.index, roll_vol_vw.values, color='#c0392b', linewidth=1.2, label='P(vw)')
ax.set_title('12-Month Rolling Volatility — Min-Variance vs Value-Weighted', fontsize=13)
ax.set_xlabel('Date')
ax.set_ylabel('Annualized Volatility')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{CHARTS_DIR}mv_vs_vw_rolling_vol.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. VW Top Holdings

In [ ]:
for Y in [Y0, Y_END]:
    if Y not in vw_weights_yearend:
        print(f'No year-end weights for {Y}')
        continue
    
    w_vw = vw_weights_yearend[Y].sort_values(ascending=False)
    top10 = w_vw.head(10)
    
    # Get market caps for display
    ym_dec = (Y, 12)
    mv_col = mv_date_map[ym_dec]
    # Convert to dict for safe .get() calls (pd.Series.get() default behavior
    # is unreliable across pandas versions when the key is not found)
    caps_dict = mv_m.loc[w_vw.index, mv_col].to_dict()
    
    print(f'\nVW Top 10 holdings at end of {Y} (for {Y+1}):')
    print(f'{"ISIN":<20} {"Name":<35} {"Weight":>8} {"MktCap ($M)":>12}')
    print('-' * 78)
    for isin, weight in top10.items():
        name = str(firm_names.get(isin, 'N/A'))[:33]
        cap = caps_dict.get(isin, 0)
        print(f'{isin:<20} {name:<35} {weight:>8.4f} {cap:>12,.0f}')
    print(f'Sum of top 10: {top10.sum():.4f}')
    print(f'Total firms with weight > 0: {(w_vw > 0).sum()}')

## 9. Save Results

In [ ]:
# VW portfolio returns
df_vw_ret.to_csv(f'{DATA_DIR}vw_portfolio_returns.csv')

# VW summary statistics
pd.DataFrame([{
    'Portfolio': 'P(vw)',
    'Ann_Return': mu_ann_vw,
    'Ann_Volatility': vol_ann_vw,
    'Sharpe_Ratio': sharpe_vw,
    'Min_Monthly_Return': np.min(r_vw),
    'Max_Monthly_Return': np.max(r_vw),
    'RF_ann': rf_ann,
}]).to_csv(f'{DATA_DIR}vw_summary_stats.csv', index=False)

# Save numerical values (not formatted strings) for downstream use
comparison_num.to_csv(f'{DATA_DIR}mv_vw_comparison.csv')

# VW year-end weights (keyed by allocation year Y, for Part II)
vw_wt_rows = []
for yr, w in vw_weights_yearend.items():
    for isin, wt in w.items():
        if wt > 1e-10:
            vw_wt_rows.append({'Year': yr, 'ISIN': isin, 'Weight': wt})
pd.DataFrame(vw_wt_rows).to_csv(f'{DATA_DIR}vw_yearend_weights.csv', index=False)

print('All results saved:')
print(f'  - {DATA_DIR}vw_portfolio_returns.csv')
print(f'  - {DATA_DIR}vw_summary_stats.csv')
print(f'  - {DATA_DIR}mv_vw_comparison.csv')
print(f'  - {DATA_DIR}vw_yearend_weights.csv')